In [120]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import pprint

In [121]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

Device: cuda


## Configuration

In [122]:
D_MODEL = 128            # Embedding size
BLOCK_SIZE = 64

## Dataset

In [123]:
data_path = Path("../../datasets/tinyshakespeare.txt")
text = data_path.read_text(encoding="utf-8")
chars = sorted(set(text))
VOCAB_SIZE = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]


def decode(ids):
    return "".join(itos[i] for i in ids)



data = torch.tensor(encode(text), dtype=torch.long)

print("Characters:", len(text))
print(pprint.pprint(stoi, compact=True))
print(f"VOCAB_SIZE = {VOCAB_SIZE}")
print("Encoded shape:", data)    

Characters: 1115393
{'\n': 0,
 ' ': 1,
 '!': 2,
 '$': 3,
 '&': 4,
 "'": 5,
 ',': 6,
 '-': 7,
 '.': 8,
 '3': 9,
 ':': 10,
 ';': 11,
 '?': 12,
 'A': 13,
 'B': 14,
 'C': 15,
 'D': 16,
 'E': 17,
 'F': 18,
 'G': 19,
 'H': 20,
 'I': 21,
 'J': 22,
 'K': 23,
 'L': 24,
 'M': 25,
 'N': 26,
 'O': 27,
 'P': 28,
 'Q': 29,
 'R': 30,
 'S': 31,
 'T': 32,
 'U': 33,
 'V': 34,
 'W': 35,
 'X': 36,
 'Y': 37,
 'Z': 38,
 'a': 39,
 'b': 40,
 'c': 41,
 'd': 42,
 'e': 43,
 'f': 44,
 'g': 45,
 'h': 46,
 'i': 47,
 'j': 48,
 'k': 49,
 'l': 50,
 'm': 51,
 'n': 52,
 'o': 53,
 'p': 54,
 'q': 55,
 'r': 56,
 's': 57,
 't': 58,
 'u': 59,
 'v': 60,
 'w': 61,
 'x': 62,
 'y': 63,
 'z': 64}
None
VOCAB_SIZE = 65
Encoded shape: tensor([18, 47, 56,  ..., 52, 45,  8])


## Tokenizer

## Simple model

In [124]:
model = nn.Sequential(
    nn.Embedding(VOCAB_SIZE, D_MODEL),
    nn.Linear(D_MODEL, VOCAB_SIZE)
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

## Training

In [129]:
model = model.to(DEVICE)

running = []
for step in range(2000):
    
    i = torch.randint(0, len(data) - BLOCK_SIZE -1 , (1,)).item()
    
    x = data[i:i+block_size].to(DEVICE)
    y = data[i+1:i+block_size+1].to(DEVICE)

    # print("x:", decode(x.tolist()))
    # print("y:", decode(y.tolist()))


    logits = model(x)

    loss = F.cross_entropy( logits, y )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running.append(loss.item())
    if step % 100 == 0:
        print(step, sum(running[-100:]) / len(running[-100:]))
    

0 2.5966567993164062
100 2.522667715549469
200 2.5063250708580016
300 2.530504970550537
400 2.5089246320724485
500 2.535075113773346
600 2.519511448144913
700 2.4926467514038086
800 2.4925453734397887
900 2.5004095458984374
1000 2.5196691405773164
1100 2.513963055610657
1200 2.51356477022171
1300 2.475403518676758
1400 2.5008563554286956
1500 2.4778433561325075
1600 2.5113480520248412
1700 2.517113713026047
1800 2.529463541507721
1900 2.4797906708717345


In [132]:
# -------------------------
# Generate
# -------------------------

x = torch.tensor(
    [stoi["A"]],
    device=DEVICE
)

model.eval()

with torch.no_grad():

    for _ in range(300):

        # x[-1:] has shape [1]
        logits = model(x[-1:])

        # If logits is [1, vocab_size],
        # take the last prediction
        logits = logits[-1]

        probs = torch.softmax(
            logits,
            dim=-1
        )

        # multinomial gives [1]
        next_token = torch.multinomial(
            probs,
            1
        )

        # Both are now [1]
        x = torch.cat([
            x,
            next_token
        ])

print("\nGenerated:\n")
print("".join(itos[i.item()] for i in x))


Generated:

ARI way, mom; ve gincathinot f af I:
keronda fass s.
Wsu rr fpin teramy onkn


QUTRKE:



Shissaie, gse, ieemer bathead-ked wikerou waithiste th' we;
Berrersse

QU;
Hcer sean horea nesape filer:
KIXleico orset id, m bbema t tanounesove JugAn ink w w. igr
MEfenusth me flllthanck londinoorrlldin,
Linse
